# Chapter 27 : Output Constrain 

Output constraints in AI are rules or boundaries set on a Large Language Model (LLM) or generative system to control the shape, length, style, or content of its response. They narrow down the "space of valid outputs" so the result is predictable, accurate, and ready for use.

1. What are output constraints?
Suppose you ask the LLM to classify a transaction's risk.
You want only three possible outputs:
- LOW
- MEDIUM
- HIGH

But the model might return:
I believe this transaction should be considered
high risk because the destination country...


That might be useful for a report, but it is inconvenient if your Python application expects a single label.
With output constraints, we try to make the response conform to a specific format.

In [3]:
from dotenv import load_dotenv
load_dotenv()
import os

groq_api_key = os.getenv("GROQ_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"])

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Simple Example

In [5]:
prompt = """
Classify the following transaction as LOW, MEDIUM, or HIGH risk.

Transaction: A customer transfers ₹500 to a known local merchant.

Rules:
- Return exactly one label.
- Allowed labels: LOW, MEDIUM, HIGH.
- Do not explain your answer.
"""

response = llm.invoke(prompt)

print(response.content)

LOW


# Lets get result in json format as we have studied in Chapter 18

In [13]:
from pydantic import BaseModel
from typing import Literal

class RiskLevel(BaseModel):
    risk_level: Literal["High", "Medium", "Low"]

response = llm.with_structured_output(RiskLevel)

prompt = """
Assess the risk level of this transaction.

Transaction: A customer transfers ₹500 to a known local merchant.
"""

try:
    result = response.invoke(prompt)
    print(result.model_dump_json(indent=2))
except Exception as e:
    print("Assessment failed:", e)

{
  "risk_level": "Low"
}


/Users/vimal/Library/Python/3.9/lib/python/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RiskLevel(risk_level='Low'), input_type=RiskLevel])
  return self.__pydantic_serializer__.to_python(
